## Classical After-Generation Watermarking Methods

To benchmark Tree-Ring watermarking—which embeds signals in the **initial noise prior to generation**—we include three widely studied **after-generation (post-hoc)** classical watermarking baselines. These methods operate directly on the **final generated image** in either the frequency or transform domains. All three are **training-free**, deterministic, and use analytical verification (no learned detectors).

---

### 1. DFT Single-Frequency Watermark

**Concept:**  
A simple but effective frequency-domain watermark that boosts the magnitude of a specific DFT (Discrete Fourier Transform) coefficient in the image’s frequency spectrum.

**Embedding:**  
Modify a known (u₀, v₀) frequency bin in the grayscale FFT:

$$
F[u_0, v_0] \leftarrow F[u_0, v_0] + \alpha \cdot \text{meanMag}
$$

**Verification:**  
Compute FFT of the test image and check the magnitude at (u₀, v₀).  
A higher magnitude indicates the presence of the watermark.

---

### 2. DWT–DCT Watermark

**Concept:**  
A multi-resolution watermark that embeds a pattern inside the **LL subband** after a DWT (Discrete Wavelet Transform) and DCT-like frequency decomposition. This exploits the stability of mid-frequency components.

**Embedding:**  
1. Apply a 1-level DWT → LL, LH, HL, HH  
2. Apply DCT-like transform (FFT2) on LL  
3. Add a watermark pattern `W` into the LL frequency patch  
4. Inverse FFT2 → inverse DWT

In 1-level 2D DWT, the image is decomposed like this:

```
+---------+---------+
|   LL    |   LH    |
| (coarse)| (edges) |
+---------+---------+
|   HL    |   HH    |
| (edges) | (fine)  |
+---------+---------+
```
**Verification:**  
Repeat DWT → DCT, extract the same patch, compute the **correlation** with the original watermark pattern.

---

### 3. DWT–DCT–SVD Watermark

**Concept:**  
An enhanced hybrid method embedding information in the **singular values** of the DCT-transformed LL subband. Singular values are stable under noise and compression, improving robustness.

**Embedding:**  
1. Apply DWT → LL  
2. Apply DCT-like transform (FFT2) on LL  
3. Perform SVD:  
$$
D = U S V^T
$$  
4. Add watermark signal to top-K singular values:  
$$
S'_i = S_i + \alpha W_i
$$  
5. Reconstruct: $U S' V^T$ → inverse DWT

**Verification:**  
Extract LL, apply DCT and SVD, compute **correlation** between extracted singular values and the known watermark vector.

---

## Summary Comparison Table

### Table 1 — High-Level Characteristics

| Method | Domain | Where It Embeds | Signal Type | Typical Robustness | Blind? |
|--------|--------|------------------|--------------|----------------------|--------|
| **DFT Single-Frequency** | Fourier | Specific (u, v) bin | Scalar magnitude boost | Weak–moderate (fragile to geometric transforms) | ✔ Fully blind |
| **DWT–DCT** | Wavelet + frequency | LL subband DCT patch | 2D pattern | Moderate (robust to slight JPEG/blur) | Semi-blind (requires W) |
| **DWT–DCT–SVD** | Wavelet + frequency + SVD | Singular values of LL | 1D vector | Stronger (stable under noise/JPEG) | Semi-blind (requires W) |

---

### Table 2 — Embedding & Verification Procedures

| Method | Embedding Procedure | Verification Procedure |
|--------|----------------------|------------------------|
| **DFT Single-Frequency** | FFT2 → boost magnitude at (u₀, v₀) → iFFT | FFT2 → read magnitude at (u₀, v₀) → threshold |
| **DWT–DCT** | DWT → 2D DCT/FFT2 → patch + W → inverse | DWT → DCT/FFT2 → extract patch → correlation(W) |
| **DWT–DCT–SVD** | DWT → DCT → SVD → modify S (top-K) → inverse | DWT → DCT → SVD → correlate S with W |

---

### Table 3 — Key Advantages and Limitations

| Method | Advantages | Limitations |
|--------|------------|-------------|
| **DFT Single-Frequency** | Simple, fast, blind detection | Easily destroyed by rotation/crop; weak hiding capacity |
| **DWT–DCT** | Good robustness vs invisibility trade-off | Requires known pattern; fragile under strong geometric distortions |
| **DWT–DCT–SVD** | Most robust; stable SVD structure | More complex; semi-blind; susceptible to large geometric warps |


## Abbreviations

| Abbreviation      | Meaning                                                  | Explanation                                                                                                                                |
| ----------------- | -------------------------------------------------------- | ------------------------------------------------------------------------------------------------------------------------------------------ |
| **DWT**           | *Discrete Wavelet Transform*                             | A transform that decomposes an image into multi-resolution subbands (LL, LH, HL, HH). Used to separate coarse structure from fine details. |
| **DCT**           | *Discrete Cosine Transform*                              | Transforms an image block into frequency components. Commonly used in JPEG; ideal for embedding watermarks in mid-frequency coefficients.  |
| **FFT**           | *Fast Fourier Transform*                                 | Efficient algorithm for computing the Discrete Fourier Transform (DFT). Used for the single-frequency watermark baseline.                  |
| **LL**            | *Low–Low subband (approximation)*                        | The coarse, downsampled version of the image obtained after DWT; stable and good for watermark embedding.                                  |
| **SVD**           | *Singular Value Decomposition*                           | Factorizes a matrix into U, S, Vᵀ; watermarking modifies the singular values (S) because they are stable under noise/compression.          |

In [ ]:
import os
import glob
import random
import time
from datetime import timedelta
from io import BytesIO
import pandas as pd
from IPython.display import display
import numpy as np
from PIL import Image, ImageFilter

import torch
from torchvision import transforms

from sklearn.metrics import roc_auc_score

try:
    import pywt
except ImportError:
    pywt = None
    print("WARNING: pywt not installed. DWT-based methods (dwt_dct, dwt_dct_svd) will not work unless you install pywt.")


# ======================
#        CONFIG
# ======================
DATA_DIR = "./watermark_dataset/clean"  # folder with clean images ONLY
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"  # not really needed but kept
SEED = 42
IMAGE_SIZE = 512

# Choose watermarking method: "dft_single", "dwt_dct", or "dwt_dct_svd"
WATERMARK_METHOD = "dwt_dct_svd"   # change to "dwt_dct_svd" or "dft_single"

# Pattern sizes / params
DWT_DCT_PATTERN_SIZE = 32
SVD_K = 32  # number of singular values used for SVD-based watermark


# ======================
#   Reproducibility
# ======================
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)


# ======================
#    Watermark helpers
# ======================

def pil_to_np_gray(img: Image.Image):
    """Convert PIL to grayscale numpy [0,1]."""
    return np.array(img.convert("L"), dtype=np.float32) / 255.0


def np_gray_to_pil(arr: np.ndarray):
    """Convert grayscale [0,1] array to RGB PIL."""
    arr = np.clip(arr * 255.0, 0, 255).astype(np.uint8)
    img_gray = Image.fromarray(arr)
    return Image.merge("RGB", [img_gray] * 3)


# ---------- DFT single-frequency watermark ----------

def apply_dft_single_frequency(img: Image.Image, strength=10.0, u0_frac=0.25, v0_frac=0.25):
    """
    Embed watermark by boosting one mid-frequency DFT coefficient in grayscale channel.
    """
    gray = img.convert("L")
    arr = np.array(gray, dtype=np.float32)

    F = np.fft.fft2(arr)
    h, w = F.shape

    u0 = int(h * u0_frac)
    v0 = int(w * v0_frac)

    mean_mag = np.abs(F).mean()
    F[u0, v0] += strength * mean_mag
    F[-u0, -v0] += strength * mean_mag  # symmetric for real-valued input

    arr_w = np.fft.ifft2(F).real
    arr_w = np.clip(arr_w, 0, 255).astype(np.uint8)
    watermarked_gray = Image.fromarray(arr_w)

    return Image.merge("RGB", [watermarked_gray] * 3)


def detect_dft_single_frequency(img: Image.Image, u0_frac=0.25, v0_frac=0.25):
    """
    Detection score: magnitude of the chosen frequency bin (higher = more likely watermarked).
    """
    gray = img.convert("L")
    arr = np.array(gray, dtype=np.float32)

    F = np.fft.fft2(arr)
    h, w = F.shape

    u0 = int(h * u0_frac)
    v0 = int(w * v0_frac)

    score = float(np.abs(F[u0, v0]))
    return score


# ---------- DWT–DCT watermark ----------

def make_dwt_dct_pattern(size=32):
    rng = np.random.RandomState(SEED)
    return rng.randn(size, size)  # zero-mean Gaussian pattern


WM_PATTERN_DWT_DCT = make_dwt_dct_pattern(DWT_DCT_PATTERN_SIZE)


def apply_dwt_dct(img: Image.Image, alpha=10.0):
    """
    Classic DWT-DCT style watermark in LL subband (using FFT2 in place of DCT for simplicity).
    """
    if pywt is None:
        raise RuntimeError("pywt is required for dwt_dct watermark.")

    arr_gray = pil_to_np_gray(img)  # [H,W] in [0,1]

    LL, (LH, HL, HH) = pywt.dwt2(arr_gray, "haar")
    dct_LL = np.fft.fft2(LL)

    h_ll, w_ll = dct_LL.shape
    hs = min(DWT_DCT_PATTERN_SIZE, h_ll)
    ws = min(DWT_DCT_PATTERN_SIZE, w_ll)

    dct_LL[:hs, :ws] += alpha * WM_PATTERN_DWT_DCT[:hs, :ws]

    LL_w = np.fft.ifft2(dct_LL).real
    coeffs2_w = (LL_w, (LH, HL, HH))
    arr_w = pywt.idwt2(coeffs2_w, "haar")

    return np_gray_to_pil(arr_w)


def detect_dwt_dct(img: Image.Image):
    """
    Detection: correlation between extracted DCT(LL) patch and watermark pattern.
    """
    if pywt is None:
        raise RuntimeError("pywt is required for dwt_dct detection.")

    arr_gray = pil_to_np_gray(img)
    LL, (LH, HL, HH) = pywt.dwt2(arr_gray, "haar")
    dct_LL = np.fft.fft2(LL).real

    h_ll, w_ll = dct_LL.shape
    hs = min(DWT_DCT_PATTERN_SIZE, h_ll)
    ws = min(DWT_DCT_PATTERN_SIZE, w_ll)

    extracted = dct_LL[:hs, :ws]
    wm = WM_PATTERN_DWT_DCT[:hs, :ws]

    x = extracted.flatten()
    y = wm.flatten()
    if np.std(x) < 1e-8 or np.std(y) < 1e-8:
        return 0.0

    corr = np.corrcoef(x, y)[0, 1]
    return float(corr)


# ---------- DWT–DCT–SVD watermark ----------

def make_svd_wm_vector(k=SVD_K):
    rng = np.random.RandomState(SEED + 1)
    return rng.randn(k)


WM_VECTOR_SVD = make_svd_wm_vector(SVD_K)


def apply_dwt_dct_svd(img: Image.Image, alpha=0.1, k=SVD_K):
    """
    DWT-DCT-SVD watermark: modify top-k singular values of DCT(LL).
    """
    if pywt is None:
        raise RuntimeError("pywt is required for dwt_dct_svd watermark.")

    arr_gray = pil_to_np_gray(img)
    LL, (LH, HL, HH) = pywt.dwt2(arr_gray, "haar")
    dct_LL = np.fft.fft2(LL).real

    U, S, Vt = np.linalg.svd(dct_LL, full_matrices=False)

    k = min(k, len(S))
    S[:k] += alpha * WM_VECTOR_SVD[:k]

    dct_LL_w = (U * S) @ Vt
    LL_w = np.fft.ifft2(dct_LL_w).real

    coeffs2_w = (LL_w, (LH, HL, HH))
    arr_w = pywt.idwt2(coeffs2_w, "haar")
    return np_gray_to_pil(arr_w)


def detect_dwt_dct_svd(img: Image.Image, k=SVD_K):
    """
    Detection: correlation between extracted top-k singular values and watermark vector.
    """
    if pywt is None:
        raise RuntimeError("pywt is required for dwt_dct_svd detection.")

    arr_gray = pil_to_np_gray(img)
    LL, (LH, HL, HH) = pywt.dwt2(arr_gray, "haar")
    dct_LL = np.fft.fft2(LL).real

    U, S, Vt = np.linalg.svd(dct_LL, full_matrices=False)
    k = min(k, len(S))
    S_sub = S[:k]
    wm_vec = WM_VECTOR_SVD[:k]

    if np.std(S_sub) < 1e-8 or np.std(wm_vec) < 1e-8:
        return 0.0

    corr = np.corrcoef(S_sub, wm_vec)[0, 1]
    return float(corr)


# Unified interface
def apply_watermark(img: Image.Image, method: str):
    if method == "dft_single":
        return apply_dft_single_frequency(img)
    elif method == "dwt_dct":
        return apply_dwt_dct(img)
    elif method == "dwt_dct_svd":
        return apply_dwt_dct_svd(img)
    else:
        raise ValueError(f"Unknown watermark method: {method}")


def detect_watermark(img: Image.Image, method: str):
    if method == "dft_single":
        return detect_dft_single_frequency(img)
    elif method == "dwt_dct":
        return detect_dwt_dct(img)
    elif method == "dwt_dct_svd":
        return detect_dwt_dct_svd(img)
    else:
        raise ValueError(f"Unknown watermark method: {method}")


# ======================
#   Attack / Augmentations
# ======================

class RandomJPEG(object):
    def __init__(self, p=1.0, q_range=(40, 70)):
        self.p = p
        self.q_low, self.q_high = q_range

    def __call__(self, img):
        if random.random() > self.p:
            return img
        buffer = BytesIO()
        q = random.randint(self.q_low, self.q_high)
        img.save(buffer, format="JPEG", quality=q)
        buffer.seek(0)
        return Image.open(buffer).convert("RGB")


def make_clean_aug(IMAGE_SIZE):
    return transforms.Resize((IMAGE_SIZE, IMAGE_SIZE))


def make_jpeg_aug(IMAGE_SIZE, q_low=40, q_high=70):
    return transforms.Compose(
        [
            transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
            RandomJPEG(p=1.0, q_range=(q_low, q_high)),
        ]
    )


def make_blur_aug(IMAGE_SIZE):
    return transforms.Compose(
        [
            transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
            transforms.Lambda(
                lambda img: img.filter(
                    ImageFilter.GaussianBlur(radius=random.uniform(1.5, 3.0))
                )
            ),
        ]
    )


def make_geom_aug(IMAGE_SIZE):
    return transforms.Compose(
        [
            transforms.RandomRotation(degrees=15),
            transforms.RandomResizedCrop(IMAGE_SIZE, scale=(0.7, 1.0)),
            transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
        ]
    )


def make_down_up_attack(IMAGE_SIZE, downscale_frac=0.5):
    small = max(1, int(IMAGE_SIZE * downscale_frac))
    return transforms.Compose(
        [
            transforms.Resize(
                (small, small), interpolation=transforms.InterpolationMode.BILINEAR
            ),
            transforms.Resize(
                (IMAGE_SIZE, IMAGE_SIZE),
                interpolation=transforms.InterpolationMode.BILINEAR,
            ),
        ]
    )


def make_msg_app_combo(IMAGE_SIZE):
    small = max(1, int(IMAGE_SIZE * 0.5))
    return transforms.Compose(
        [
            transforms.Resize(
                (small, small), interpolation=transforms.InterpolationMode.BILINEAR
            ),
            RandomJPEG(p=1.0, q_range=(40, 70)),
            transforms.Resize(
                (IMAGE_SIZE, IMAGE_SIZE),
                interpolation=transforms.InterpolationMode.BILINEAR,
            ),
        ]
    )


def make_random_crop_attack(IMAGE_SIZE, scale=(0.5, 0.9)):
    return transforms.Compose(
        [
            transforms.RandomResizedCrop(IMAGE_SIZE, scale=scale, ratio=(0.75, 1.33)),
        ]
    )


def make_occlusion_block(IMAGE_SIZE, box_frac=0.25):
    class Block(object):
        def __init__(self, frac):
            self.frac = frac

        def __call__(self, img):
            w, h = img.size
            bw, bh = int(w * self.frac), int(h * self.frac)
            x0 = random.randint(0, max(0, w - bw))
            y0 = random.randint(0, max(0, h - bh))
            img = img.copy()
            import PIL.ImageDraw as ImageDraw

            draw = ImageDraw.Draw(img)
            draw.rectangle([x0, y0, x0 + bw, y0 + bh], fill=(0, 0, 0))
            return img

    return transforms.Compose(
        [transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)), Block(box_frac)]
    )


attack_factories = {
    "clean":        lambda: make_clean_aug(IMAGE_SIZE),
    "jpeg_strong":  lambda: make_jpeg_aug(IMAGE_SIZE, q_low=40, q_high=60),
    "msg_app_combo":lambda: make_msg_app_combo(IMAGE_SIZE),
    "down_up":      lambda: make_down_up_attack(IMAGE_SIZE, downscale_frac=0.5),
    "blur":         lambda: make_blur_aug(IMAGE_SIZE),
    "random_crop":  lambda: make_random_crop_attack(IMAGE_SIZE, scale=(0.5, 0.9)),
    "occlusion":    lambda: make_occlusion_block(IMAGE_SIZE, box_frac=0.25),
    "geom_warp":    lambda: make_geom_aug(IMAGE_SIZE),
}


# ======================
#       EVAL UTILS
# ======================

def compute_best_accuracy(scores, labels):
    """
    Sweep thresholds over sorted scores to find max accuracy.
    """
    scores = np.asarray(scores, dtype=np.float64)
    labels = np.asarray(labels, dtype=np.int32)

    uniq = np.unique(scores)
    if len(uniq) == 1:
        # degenerate case
        return float((labels == (scores >= uniq[0]).astype(int)).mean()), float(uniq[0])

    best_acc = 0.0
    best_thr = uniq[0]

    for thr in uniq:
        preds = (scores >= thr).astype(np.int32)
        acc = (preds == labels).mean()
        if acc > best_acc:
            best_acc = acc
            best_thr = thr

    return float(best_acc), float(best_thr)


def avg(values):
    return sum(values) / len(values) if values else float("nan")


# ======================
#         MAIN
# ======================

# Collect clean images
exts = ("*.png", "*.jpg", "*.jpeg", "*.bmp")
file_paths = []
for e in exts:
    file_paths.extend(glob.glob(os.path.join(DATA_DIR, e)))
file_paths = sorted(file_paths)
if not file_paths:
    raise RuntimeError(f"No images found in {DATA_DIR}")
print(f"Found {len(file_paths)} clean images in {DATA_DIR}")
print(f"Watermark method: {WATERMARK_METHOD}")

testing_times = 3
total_start = time.time()

resize_base = transforms.Resize((IMAGE_SIZE, IMAGE_SIZE))
results = []

for attack_name, aug_builder in attack_factories.items():
    print("\n" + "#" * 80)
    print(f"# ATTACK: {attack_name}")
    print("#" * 80)

    attack_aug = aug_builder()

    acc_all, auc_all = [], []

    for test_i in range(testing_times):
        iter_start = time.time()
        print("=" * 80)
        print(
            f"[{attack_name}] Test {test_i + 1:3d}/{testing_times:3d}    "
            f"Time elapsed: {str(timedelta(seconds=int(time.time() - total_start)))}"
        )
        print("-" * 80)

        scores = []
        labels = []

        # Build a (clean, watermarked) pair for each image
        for path in file_paths:
            img = Image.open(path).convert("RGB")
            img = resize_base(img)

            # clean example
            img_clean = attack_aug(img)
            score_clean = detect_watermark(img_clean, WATERMARK_METHOD)
            scores.append(score_clean)
            labels.append(0)

            # watermarked example
            img_wm = apply_watermark(img, WATERMARK_METHOD)
            img_wm = attack_aug(img_wm)
            score_wm = detect_watermark(img_wm, WATERMARK_METHOD)
            scores.append(score_wm)
            labels.append(1)

        labels_np = np.asarray(labels, dtype=np.int32)
        scores_np = np.asarray(scores, dtype=np.float64)

        # AUROC
        try:
            auc = roc_auc_score(labels_np, scores_np)
        except ValueError:
            auc = float("nan")

        # Best accuracy via threshold sweep
        best_acc, best_thr = compute_best_accuracy(scores_np, labels_np)

        acc_all.append(best_acc)
        auc_all.append(auc)

        iter_time = time.time() - iter_start
        col1_w = 14
        col_w = 18
        print(f"{'Metric':<{col1_w}} {'Value':>{col_w}}")
        print("-" * (col1_w + col_w + 3))
        print(f"{'Best_Acc':<{col1_w}} {best_acc:>{col_w}.4f}")
        print(f"{'Best_Thr':<{col1_w}} {best_thr:>{col_w}.4f}")
        print(f"{'AUROC':<{col1_w}}   {auc:>{col_w}.4f}")
        print("-" * (col1_w + col_w + 3))
        print(f"Iter time: {iter_time:.1f}s")
        print("=" * 80)

    # store averages for this attack
    results.append({
        "attack": attack_name,
        "best_acc_mean": avg(acc_all),
        "auroc_mean": avg(auc_all),
    })

print("\n" + "=" * 80)
print("All evaluations complete.")
print("=" * 80)

df_results = pd.DataFrame(results)
display(df_results)

Found 500 clean images in ./watermark_dataset/clean
Watermark method: dwt_dct_svd

################################################################################
# ATTACK: clean
################################################################################
[clean] Test   1/  3    Time elapsed: 0:00:00
--------------------------------------------------------------------------------
Metric                      Value
-----------------------------------
Best_Acc                   0.5160
Best_Thr                   0.0059
AUROC                        0.5089
-----------------------------------
Iter time: 38.7s
[clean] Test   2/  3    Time elapsed: 0:00:38
--------------------------------------------------------------------------------
Metric                      Value
-----------------------------------
Best_Acc                   0.5160
Best_Thr                   0.0059
AUROC                        0.5089
-----------------------------------
Iter time: 38.1s
[clean] Test   3/  3    Time el

,attack,best_acc_mean,auroc_mean
0,clean,0.516000,0.508936
1,jpeg_strong,0.515667,0.507505
2,msg_app_combo,0.510667,0.504787
3,down_up,0.512000,0.506704
4,blur,0.509667,0.504463
5,random_crop,0.626667,0.662937
6,occlusion,0.529333,0.514609
7,geom_warp,0.558333,0.567936


: 

##  dft_single
| attack       | best_acc_mean | auroc_mean |
|--------------|---------------|------------|
| clean        | 1.000000      | 1.000000   |
| jpeg_strong  | 0.710667      | 0.768173   |
| msg_app_combo| 0.503333      | 0.456591   |
| down_up      | 0.537000      | 0.528172   |
| blur         | 0.638333      | 0.613409   |
| random_crop  | 0.508333      | 0.492869   |
| occlusion    | 1.000000      | 1.000000   |
| geom_warp    | 0.520333      | 0.494442   |

## dwt_dct

| attack        | best_acc_mean | auroc_mean |
|---------------|---------------|------------|
| clean         | 0.636000      | 0.659032   |
| jpeg_strong   | 0.627667      | 0.649775   |
| msg_app_combo | 0.611667      | 0.630228   |
| down_up       | 0.629000      | 0.647620   |
| blur          | 0.616000      | 0.632211   |
| random_crop   | 0.513667      | 0.497580   |
| occlusion     | 0.609333      | 0.632967   |
| geom_warp     | 0.512667      | 0.494796   |

## dwt_dct_svd
| attack        | best_acc_mean | auroc_mean |
|---------------|---------------|------------|
| clean         | 0.516000      | 0.508936   |
| jpeg_strong   | 0.515667      | 0.507505   |
| msg_app_combo | 0.510667      | 0.504787   |
| down_up       | 0.512000      | 0.506704   |
| blur          | 0.509667      | 0.504463   |
| random_crop   | 0.626667      | 0.662937   |
| occlusion     | 0.529333      | 0.514609   |
| geom_warp     | 0.558333      | 0.567936   |
